# Morris 28-Gene TSS-Centered Enformer Embedding Generation

**Purpose**: Generate Enformer trunk embeddings for 28 target genes in the Morris STINGseq CRISPRi dataset

**Important**: All 28 genes are generated fresh (existing pilot_full_v2.h5 is enhancer-centered and cannot be used)

**Estimated time**: ~30 minutes (T4 GPU)

## 1. Setup

In [ ]:
# Step 1: Install packages
# Note: Restart runtime after running this cell
!pip install -q numpy==1.26.4 scipy==1.13.0
!pip install -q enformer-pytorch pyfaidx h5py

print("\n" + "="*60)
print("Installation complete!")
print("")
print(">>> Runtime -> Restart runtime <<<")
print("")
print("After restart, skip this cell and start from Step 2")
print("="*60)

In [ ]:
# Step 2: Start here after runtime restart
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Data path configuration
DRIVE_BASE = '/content/drive/MyDrive/cdt_data'

# Verify required files
!ls -la {DRIVE_BASE}/

## 2. Download hg38 Reference Genome

**Note**: ~3GB download

In [ ]:
%%time
import os

# Download hg38 if not already present
if not os.path.exists('/content/hg38.fa'):
    print('Downloading hg38.fa...')
    !wget -q https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz
    !gunzip hg38.fa.gz
    print('Done!')
else:
    print('hg38.fa already exists')

## 3. Load TSS Coordinate Data

**Note**: All 28 genes are generated fresh

In [ ]:
# Load TSS coordinate file
# TSS coordinates are included in the repository under data/
import pandas as pd
import os

# Auto-detect environment
if os.path.exists('/content'):
    # Google Colab: upload or clone the repo first
    TSS_PATH = '/content/CDT2/data/morris_28genes_tss.csv'
    if not os.path.exists(TSS_PATH):
        # Fallback: try Google Drive
        TSS_PATH = f'{DRIVE_BASE}/morris_28genes_tss.csv'
else:
    # Local: relative to repo root
    TSS_PATH = '../../data/morris_28genes_tss.csv'

tss_df = pd.read_csv(TSS_PATH)
print(f'Total genes to generate: {len(tss_df)}')
print()
tss_df[['gene_name', 'chrom', 'tss', 'strand']]

## 4. Compute Enformer Embeddings

In [ ]:
import h5py
import numpy as np
from tqdm import tqdm
import torch
from pathlib import Path

# Constants
ENFORMER_SEQ_LENGTH = 196_608
ENFORMER_OUTPUT_BINS = 896
ENFORMER_TRUNK_DIM = 3072

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

In [ ]:
# Load Enformer model
from enformer_pytorch import Enformer

print('Loading Enformer model...')
model = Enformer.from_pretrained('EleutherAI/enformer-official-rough')
model = model.to(DEVICE)
model.eval()
print('Model loaded!')

In [ ]:
# Load reference genome
from pyfaidx import Fasta

print('Loading genome...')
genome = Fasta('/content/hg38.fa')
print('Genome loaded!')

In [ ]:
# Helper functions
def one_hot_encode(sequence):
    """One-hot encode DNA sequence (A, C, G, T -> 4 channels)."""
    mapping = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    seq_upper = sequence.upper()
    encoded = np.zeros((len(sequence), 4), dtype=np.float32)
    for i, base in enumerate(seq_upper):
        if base in 'ACGT':
            encoded[i, mapping[base]] = 1.0
    return encoded

def extract_sequence(chrom, center):
    """Extract 196kb sequence centered at TSS."""
    half_len = ENFORMER_SEQ_LENGTH // 2
    start = max(0, center - half_len)
    end = center + half_len
    
    # Handle chromosome naming (chr1 vs 1)
    if chrom not in genome.keys():
        if chrom.startswith('chr'):
            chrom_alt = chrom[3:]
        else:
            chrom_alt = f'chr{chrom}'
        if chrom_alt in genome.keys():
            chrom = chrom_alt
    
    chrom_len = len(genome[chrom])
    if end > chrom_len:
        seq = str(genome[chrom][start:chrom_len])
        seq += 'N' * (end - chrom_len)
    else:
        seq = str(genome[chrom][start:end])
    
    if start < 0:
        seq = 'N' * (-start) + seq
    
    return seq

def run_enformer(sequence):
    """Run Enformer and extract trunk embeddings."""
    with torch.no_grad():
        encoded = one_hot_encode(sequence)
        batch_tensor = torch.from_numpy(encoded).unsqueeze(0).to(DEVICE)
        output = model(batch_tensor, return_only_embeddings=True)
        return output.cpu().numpy()[0]

In [ ]:
%%time
# Main computation loop - generate all 28 genes
print(f'Processing ALL {len(tss_df)} genes...')

embeddings = []
processed_genes = []
failed_genes = []

for i, row in tqdm(tss_df.iterrows(), total=len(tss_df), desc='Computing'):
    gene_name = row['gene_name']
    chrom = row['chrom']
    tss = row['tss']
    
    try:
        seq = extract_sequence(chrom, tss)
        emb = run_enformer(seq)
        embeddings.append(emb)
        processed_genes.append({
            'gene_name': gene_name,
            'chrom': chrom,
            'tss': tss,
            'strand': row['strand'],
            'ensembl_id': row['ensembl_id']
        })
        print(f'✓ {gene_name}')
    except Exception as e:
        print(f'✗ {gene_name}: {e}')
        failed_genes.append(gene_name)
    
    # Clear GPU cache periodically
    if (len(embeddings)) % 10 == 0:
        torch.cuda.empty_cache()

print(f'\nDone! Processed: {len(processed_genes)}, Failed: {len(failed_genes)}')
if failed_genes:
    print(f'Failed genes: {failed_genes}')

## 5. Save

In [ ]:
# Convert embeddings to numpy array
all_embeddings = np.array(embeddings)
print(f'Total embeddings: {len(all_embeddings)}')
print(f'Shape: {all_embeddings.shape}')

In [ ]:
# Save as H5 file
OUTPUT_PATH = f'{DRIVE_BASE}/morris_28genes_enformer.h5'

with h5py.File(OUTPUT_PATH, 'w') as f:
    # Embeddings [28, 896, 3072]
    f.create_dataset('embeddings', data=all_embeddings, compression='gzip', compression_opts=4)
    
    # Gene information
    gene_names = [g['gene_name'] for g in processed_genes]
    f.create_dataset('gene_names', data=np.array(gene_names, dtype='S'))
    
    chroms = [g['chrom'] for g in processed_genes]
    f.create_dataset('chroms', data=np.array(chroms, dtype='S'))
    
    tss_coords = [g['tss'] for g in processed_genes]
    f.create_dataset('tss_coords', data=np.array(tss_coords, dtype=np.int64))
    
    strands = [g['strand'] for g in processed_genes]
    f.create_dataset('strands', data=np.array(strands, dtype='S'))
    
    ensembl_ids = [g['ensembl_id'] for g in processed_genes]
    f.create_dataset('ensembl_ids', data=np.array(ensembl_ids, dtype='S'))
    
    # Metadata
    f.attrs['n_genes'] = len(processed_genes)
    f.attrs['n_bins'] = ENFORMER_OUTPUT_BINS
    f.attrs['embedding_dim'] = ENFORMER_TRUNK_DIM
    f.attrs['seq_length'] = ENFORMER_SEQ_LENGTH
    f.attrs['description'] = 'Morris 28 target genes TSS-centered Enformer embeddings for CDT v2'

print(f'Saved to: {OUTPUT_PATH}')

In [ ]:
# Verify output
with h5py.File(OUTPUT_PATH, 'r') as f:
    print('File contents:')
    for key in f.keys():
        data = f[key]
        if hasattr(data, 'shape'):
            print(f'  {key}: {data.shape}')
        else:
            print(f'  {key}: {len(data)}')
    print()
    print('Attributes:')
    for key, val in f.attrs.items():
        print(f'  {key}: {val}')
    print()
    print('Gene names:')
    gene_names = [g.decode() if isinstance(g, bytes) else g for g in f['gene_names'][:]]
    print(f'  {gene_names}')

In [ ]:
# Check file size
import os
size_mb = os.path.getsize(OUTPUT_PATH) / (1024 * 1024)
print(f'File size: {size_mb:.1f} MB')

## 6. Done

Generated file:
- `/content/drive/MyDrive/cdt_data/morris_28genes_enformer.h5`

**Structure**:
```
embeddings: [28, 896, 3072]  # Enformer trunk embeddings
gene_names: [28]              # Gene symbols
chroms: [28]                  # Chromosome names
tss_coords: [28]              # TSS positions
strands: [28]                 # Strand (+/-)
ensembl_ids: [28]             # Ensembl gene IDs
```

Use this file for CDT-II training.

**Note**: CRISPRi effects are computed for 27 genes (excluding CD55).